# YOLO-IOD: Visualization of Failure Cases
Notebook này dùng để trực quan hóa các lỗi thường gặp trong quá trình nhận diện: Misclassification, Missing, False Detection (giống Figure 9 của STAR-IOD).

## 1. Environment Setup (Từ file dota_run_colab_dota)
Chạy các cell dưới đây để cài đặt môi trường giống hệt lúc train, bao gồm cả các bản vá lỗi (fix bug).

In [ ]:
!/content/miniconda/envs/yoloiod/bin/pip install albumentations

In [ ]:
!/content/miniconda/envs/yoloiod/bin/pip install "numpy<2"

In [ ]:
import os
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/Miniconda3-latest-Linux-x86_64.sh
!bash /content/Miniconda3-latest-Linux-x86_64.sh -b -p /content/miniconda

!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!/content/miniconda/bin/conda create -n yoloiod python=3.10 -y

os.environ["PATH"] = "/content/miniconda/envs/yoloiod/bin:" + os.environ["PATH"]
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Install golden versions of dependencies inside the Conda environment
!/content/miniconda/envs/yoloiod/bin/pip install setuptools==69.5.1
!/content/miniconda/envs/yoloiod/bin/pip install torch==2.0.0 torchvision==0.15.1 --index-url https://download.pytorch.org/whl/cu118
!/content/miniconda/envs/yoloiod/bin/pip install -U openmim

!/content/miniconda/envs/yoloiod/bin/mim install "mmengine>=0.10.3"
!/content/miniconda/envs/yoloiod/bin/mim install "mmcv==2.0.1"
!/content/miniconda/envs/yoloiod/bin/mim install "mmdet==3.1.0"

!/content/miniconda/envs/yoloiod/bin/pip install -r requirements/basic_requirements.txt
!/content/miniconda/envs/yoloiod/bin/pip install "numpy<2" transformers==4.30.2 scikit-learn prettytable

# Build mmyolo and the main project using the Conda environment
!cd third_party/mmyolo && /content/miniconda/envs/yoloiod/bin/pip install -v -e . --no-build-isolation
!/content/miniconda/envs/yoloiod/bin/pip install -v -e . --no-build-isolation

In [ ]:
import os

# Sử dụng đường dẫn tương đối để tránh lỗi sai đường dẫn Drive tuyệt đối
file_path = 'yolo_world/datasets/transformers/mm_transforms.py'

if not os.path.exists(file_path):
    print(f"❌ Không tìm thấy file {file_path}. Bạn đang chạy cell ở thư mục nào?")
    print("Thư mục hiện tại:", os.getcwd())
else:
    print(f"🔍 Tìm thấy file: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 1. Định nghĩa đoạn mã lỗi gốc
    target_unknown = """        if 'instances' in results:
            retaged_instances = []
            for idx, inst in enumerate(results['instances']):
                label = inst['bbox_label']
                if label in label2ids:
                    inst['bbox_label'] = label2ids[label]
                    retaged_instances.append(inst)
            results['instances'] = retaged_instances"""

    # 2. Định nghĩa đoạn mã sửa
    all_label_mapping_block = """        for idx, label in enumerate(results[gt_label_tag]):
            results[gt_label_tag][idx] = all_label_to_idx[label]"""

    replacement_all_label_mapping_block = """        for idx, label in enumerate(results[gt_label_tag]):
            results[gt_label_tag][idx] = all_label_to_idx[label]

        if 'instances' in results:
            retaged_instances = []
            for idx, inst in enumerate(results['instances']):
                label = inst['bbox_label']
                if label in label2ids:
                    mapped_label = label2ids[label]
                    inst['bbox_label'] = all_label_to_idx[mapped_label]
                    retaged_instances.append(inst)
            results['instances'] = retaged_instances"""

    # Áp dụng vá lỗi
    if target_unknown in content:
        # Xóa khối mapping cũ ở đầu
        content = content.replace(target_unknown, "", 1)
        # Thay thế khối mapping mới sau all_label_to_idx
        content = content.replace(all_label_mapping_block, replacement_all_label_mapping_block, 1)
        print("-> Vá RandomLoadTextUnknown thành công!")

        # Làm tương tự lần 2 cho RandomLoadTextSep
        if target_unknown in content:
            content = content.replace(target_unknown, "", 1)
            content = content.replace(all_label_mapping_block, replacement_all_label_mapping_block, 1)
            print("-> Vá RandomLoadTextSep thành công!")

        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        print("🎉 Đã vá và lưu thành công file mm_transforms.py!")
    else:
        # Kiểm tra nếu đã vá rồi
        if "all_label_to_idx[mapped_label]" in content:
            print("👍 File mm_transforms.py đã được vá từ trước rồi!")
        else:
            print("⚠️ Không tìm thấy đoạn code cần vá. Có thể cấu trúc file đã thay đổi.")

In [ ]:
import os
print("Môi trường đã thiết lập xong. Chuẩn bị chạy visualization!")


In [ ]:
%%writefile run_failures_vis.py
import os
import cv2
import matplotlib.pyplot as plt
import argparse
from mmdet.apis import init_detector, inference_detector
from mmyolo.utils import register_all_modules

# Register modules của MMYolo
register_all_modules()

def visualize_failures(config_file, checkpoint_file, failure_images, output_dir):
    import torch
    model = init_detector(config_file, checkpoint_file, device="cuda:0" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    failure_types = ["Misclassification", "Missing", "False Detection"]
    
    for i, img_path in enumerate(failure_images):
        if not os.path.exists(img_path):
            print(f"Warning: {img_path} not found.")
            axes[i].axis("off")
            axes[i].set_title(f"{failure_types[i]} (Not Found)")
            continue
        result = inference_detector(model, img_path)
        
        from mmdet.registry import VISUALIZERS
        visualizer = VISUALIZERS.build(model.cfg.visualizer)
        visualizer.dataset_meta = model.dataset_meta
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        visualizer.add_datasample(
            "result",
            img,
            data_sample=result,
            draw_gt=False,
            show=False,
            pred_score_thr=0.25 # Hạ threshold để dễ quan sát lỗi false detection
        )
        drawn_img = visualizer.get_image()
        
        axes[i].imshow(drawn_img)
        axes[i].axis("off")
        axes[i].set_title(failure_types[i], fontsize=14, fontweight="bold")
        
    plt.tight_layout()
    out_path = os.path.join(output_dir, "Failure_Cases.png")
    plt.savefig(out_path, dpi=300)
    print(f"Saved visualization to {out_path}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--img-miscls", required=True)
    parser.add_argument("--img-missing", required=True)
    parser.add_argument("--img-false", required=True)
    parser.add_argument("--output-dir", required=True)
    args = parser.parse_args()
    
    visualize_failures(
        args.config, 
        args.checkpoint, 
        [args.img_miscls, args.img_missing, args.img_false], 
        args.output_dir
    )


## Chạy thử với DOTA Task 2 (tìm lỗi Helicopter)

In [ ]:
# Cấu hình đường dẫn
CONFIG_DOTA = "configs/dota_5_5_5/yolo_iod_dota_5_5_5_task2.py"
CKPT_DOTA = "work_dirs/yolo_iod_dota_5_5_5_task2/epoch_20.pth"
# Hãy đổi các đường dẫn ảnh dưới đây sang ảnh thực tế trên Google Drive của bạn
IMG_MISCLS = "data_drive/DOTA/images/test/P0001.jpg"
IMG_MISSING = "data_drive/DOTA/images/test/P0002.jpg"
IMG_FALSE = "data_drive/DOTA/images/test/P0003.jpg"
OUTPUT_DIR = "outputs/dota_failures"

# Chạy inference thông qua Python script chạy trong conda env yoloiod
!MPLBACKEND=agg PYTHONPATH="$(pwd):$(pwd)/third_party/mmyolo" /content/miniconda/envs/yoloiod/bin/python run_failures_vis.py \
    --config {CONFIG_DOTA} \
    --checkpoint {CKPT_DOTA} \
    --img-miscls {IMG_MISCLS} \
    --img-missing {IMG_MISSING} \
    --img-false {IMG_FALSE} \
    --output-dir {OUTPUT_DIR}

# Hiển thị ảnh kết quả
import os
from IPython.display import Image, display
fail_img_path = os.path.join(OUTPUT_DIR, "Failure_Cases.png")
if os.path.exists(fail_img_path):
    display(Image(fail_img_path))
else:
    print("❌ Không tìm thấy ảnh kết quả. Hãy kiểm tra xem các đường dẫn ảnh đầu vào và model checkpoint đã đúng chưa và chạy lại cell này.")
